# Analyse de profilage des datasets d'accidents 2024




In [6]:
import pandas as pd
from pathlib import Path

def load_csv(filename):
    path = Path(filename)
    return pd.read_csv(path, sep=';', quotechar='"', encoding='utf-8', dtype=str).replace({'': pd.NA, 'N/A': pd.NA, ' -1': pd.NA, ' -1 ': pd.NA})

files = [
    'caract-2024.csv',
    'lieux-2024.csv',
    'usagers-2024.csv',
    'vehicules-2024.csv'
]
datasets = {f: load_csv(f) for f in files}
for name, df in datasets.items():
    print(f'{name}: {df.shape[0]} lignes, {df.shape[1]} colonnes')


caract-2024.csv: 54402 lignes, 15 colonnes
lieux-2024.csv: 70248 lignes, 18 colonnes
usagers-2024.csv: 125187 lignes, 16 colonnes
vehicules-2024.csv: 92678 lignes, 11 colonnes


## A. Dataset Structure



### 1. caract-2024.csv
Ce fichier décrit les caractéristiques générales de l accident. Il contient 15 colonnes.
- Num_Acc : identifiant unique de l’accident, c'est la clé principale pour relier les fichiers.
- jour, mois, an : date de l’accident, ce sont des champs numériques qui permettent des analyses temporelles.
- hrmn : heure et minute au format HH:MM. Utile pour vérifier les pics horaires.
- lum : état lumineux, un code qui décrit la visibilité (jour, nuit, etc.).
- dep, com : code du département et de la commune, utiles pour la géographie administrative.
- agg : présence ou absence d’agglomération, ce qui influence le contexte urbain/rural.
- int : indication d’intersection, donc un aspect de la géométrie du lieu.
- atm : conditions atmosphériques (pluie, brouillard, etc.).
- col : type de collision, très important pour étudier le mécanisme de l’accident.
- adr : adresse textuelle, qui peut être utilisée pour cartographier même sans coordonnées exactes.
- lat, long : latitude et longitude, essentiels pour les analyses spatiales et la visualisation sur carte.


### 2. lieux-2024.csv
Ce fichier précise le lieu et les caractéristiques de la route. Il est plus technique.
- Num_Acc : permet de relier au fichier d accident.
- catr : catégorie de route / lieu.
- voie : nom de la voie ou description du site.
- v1, v2 : sens de circulation.
- circ : type de circulation sur cette section.
- nbv : nombre de voies, important pour l’infrastructure.
- vosp : présence d’une voie spéciale ou bus, utile pour le profil de la route.
- prof, pr, pr1, plan : paramètres géométriques et topographiques du lieu.
- lartpc, larrout : séparations latérales et protections.
- surf : état de la surface de la route.
- infra : présence d’infrastructure (pont, tunnel, etc.).
- situ : situation spécifique du lieu.
- vma : vitesse maximale autorisée sur le tronçon.


### 3. usagers-2024.csv
Ce fichier concerne les personnes impliquées dans l accident.
- Num_Acc : lien vers l accident principal.
- id_usager : identifiant unique de l’usager.
- id_vehicule : identifiant du véhicule associé.
- num_veh : numéro du véhicule dans l accident.
- place : position à l intérieur du véhicule (conducteur, passager, etc.).
- catu : catégorie de l usager (conducteur, piéton, cycliste, etc.).
- grav : gravité des blessures, essentiel pour l analyse des conséquences.
- sexe : sexe du participant.
- an_nais : année de naissance, qui permet de calculer l’âge.
- trajet : motif du trajet.
- secu1, secu2, secu3 : équipements de protection utilisés.
- locp, actp, etatp : localisation et état final de l usager.


### 4. vehicules-2024.csv
Ce fichier décrit les véhicules impliqués.
- Num_Acc : identifiant d accident.
- id_vehicule : identifiant du véhicule.
- num_veh : numéro du véhicule dans l accident.
- senc : sens de circulation ou position du véhicule.
- catv : catégorie de véhicule.
- obs : observations complémentaires.
- obsm : autre mention d observation.
- choc : zone de choc sur le véhicule.
- manv : manoeuvre en cours au moment de l accident.
- motor : motorisation.
- occutc : nombre d occupants du véhicule.


### structure général
Globalement, caract-2024.csv est le fichier principal pour l accident lui-même, lieux-2024.csv est le contexte routier, usagers-2024.csv traite des personnes, et vehicules-2024.csv décrit les véhicules.
En combinant ces quatre fichiers, on peut faire une analyse complète : date/heure, lieu, profils des victimes, et caractéristiques des véhicules.

## B. Missing Values and Completeness


In [7]:
def missing_summary(df):
    missing = df.isna().sum()
    pct = (missing / len(df) * 100).round(2)
    return pd.DataFrame({'missing_count': missing, 'missing_pct': pct})

for name, df in datasets.items():
    print(f'### {name}')
    display(missing_summary(df))


### caract-2024.csv


,missing_count,missing_pct
Num_Acc,0,0.00
jour,0,0.00
mois,0,0.00
an,0,0.00
hrmn,0,0.00
lum,0,0.00
dep,0,0.00
com,0,0.00
agg,0,0.00
int,0,0.00


### lieux-2024.csv


,missing_count,missing_pct
Num_Acc,0,0.00
catr,0,0.00
voie,13331,18.98
v1,16272,23.16
v2,64332,91.58
circ,4354,6.20
nbv,4178,5.95
vosp,3832,5.45
prof,50,0.07
pr,27364,38.95


### usagers-2024.csv


,missing_count,missing_pct
Num_Acc,0,0.00
id_usager,0,0.00
id_vehicule,0,0.00
num_veh,0,0.00
place,3,0.00
catu,0,0.00
grav,0,0.00
sexe,2395,1.91
an_nais,2579,2.06
trajet,2626,2.10


### vehicules-2024.csv


,missing_count,missing_pct
Num_Acc,0,0.00
id_vehicule,0,0.00
num_veh,0,0.00
senc,68,0.07
catv,1,0.00
obs,27,0.03
obsm,30,0.03
choc,44,0.05
manv,27,0.03
motor,192,0.21


### Analyse des valeurs manquantes
En regardant l ensemble des datasets, on observe des niveaux de complétude très différents.
- Dans caract-2024.csv, la plupart des colonnes sont complètes, à l exception de adr et col. Cela signifie que les analyses temporelles et géographiques sont assez fiables, mais les études basées sur le type de collision ou l adresse textuelle seront plus fragiles.
- Dans lieux-2024.csv, plusieurs colonnes techniques telles que lartpc, larrout, pr, pr1 et voie sont très souvent manquantes. Cela rend l interprétation géométrique de la route moins robuste.
- Dans usagers-2024.csv, les colonnes démographiques et de sécurité comme sexe, an_nais, locp, actp et surtout etatp sont souvent vides. Cela limite fortement les analyses sur le profil des victimes et leurs conséquences.
- Dans vehicules-2024.csv, occutc est quasiment toujours vide, donc il ne faut pas compter sur ce champ pour estimer directement le nombre d occupants.

Les manques les plus problématiques sont donc : etatp et occutc pour estimer les conséquences, lartpc/larrout pour le contexte de route, et adr / voie pour la localisation textuelle.

### Remédiation proposées

- Pour caract-2024.csv : utiliser lat et long pour remplacer partiellement adr, puis exclure les observations sans col des analyses sur le type de collision.
- Pour lieux-2024.csv : se concentrer sur les colonnes bien renseignées (catr, circ, nbv, vma) et traiter les colonnes très vides comme des informations secondaires.
- Pour usagers-2024.csv : utiliser catu et grav comme variables principales si sexe et an_nais sont manquants, et documenter clairement les limites de etatp.
- Pour vehicules-2024.csv : compléter la description des occupants via la table usagers, car occutc n est pas exploitable.


## C. Consistency and Validity Checks


In [8]:
print('--- Vérification des doublons ---')
for name, df in datasets.items():
    dup_all = df.duplicated().sum()
    if name == 'caract-2024.csv':
        dup_key = df.duplicated(subset=['Num_Acc']).sum()
    elif name == 'lieux-2024.csv':
        dup_key = df.duplicated(subset=['Num_Acc', 'voie']).sum()
    elif name == 'usagers-2024.csv':
        dup_key = df.duplicated(subset=['Num_Acc', 'id_usager']).sum()
    else:
        dup_key = df.duplicated(subset=['Num_Acc', 'id_vehicule']).sum()
    print(f'{name}: doublons totaux={dup_all}, doublons clefs={dup_key}')
print('')

print('--- Vérification des plages de valeurs ---')
df = datasets['caract-2024.csv']
print('hrmn valid count :', df['hrmn'].dropna().str.match(r'^(?:[0-1][0-9]|2[0-3]):[0-5][0-9]$').sum(), '/', len(df))
print('lat/long valides :', ((df['lat'].dropna().str.replace(',', '.').astype(float).between(-90, 90)) & (df['long'].dropna().str.replace(',', '.').astype(float).between(-180, 180))).sum(), '/', len(df))
print('an_nais min/max :', pd.to_numeric(datasets['usagers-2024.csv']['an_nais'], errors='coerce').min(), pd.to_numeric(datasets['usagers-2024.csv']['an_nais'], errors='coerce').max())
print('vma invalides :', datasets['lieux-2024.csv'].loc[datasets['lieux-2024.csv']['vma'].notna() & (~datasets['lieux-2024.csv']['vma'].str.match(r'^\d+$')), 'vma'].unique().tolist())

print('--- Anomalies catégorielles ---')
for name, col in [('caract-2024.csv', 'col'), ('usagers-2024.csv', 'sexe'), ('usagers-2024.csv', 'grav'), ('vehicules-2024.csv', 'catv'), ('lieux-2024.csv', 'catr')]:
    vals = datasets[name][col].value_counts(dropna=False).head(20)
    print(f'{name}.{col}:')
    print(vals.to_string())
    print('')


--- Vérification des doublons ---
caract-2024.csv: doublons totaux=0, doublons clefs=0
lieux-2024.csv: doublons totaux=2, doublons clefs=1831
usagers-2024.csv: doublons totaux=0, doublons clefs=0
vehicules-2024.csv: doublons totaux=0, doublons clefs=0

--- Vérification des plages de valeurs ---
hrmn valid count : 54402 / 54402
lat/long valides : 54402 / 54402
an_nais min/max : 1914.0 2024.0
vma invalides : ['90', '30', '50', '70', '20', '80', '25', '110', '130', '15', '10', '45', '5', '6', '40', '1', '3', '60', '500', '300', '100', '75', '2', '85', '35', '95', '800', '55', '140', '4', '700', '0', '900', '16', '301']
--- Anomalies catégorielles ---
caract-2024.csv.col:
col
3       16371
6       15968
2        7162
1        6057
7        5463
4        1821
5        1554
<NA>        6

usagers-2024.csv.sexe:
sexe
1       83864
2       38928
<NA>     2395

usagers-2024.csv.grav:
grav
1    52920
4    49709
3    19126
2     3432

vehicules-2024.csv.catv:
catv
7     53911
33     7777
10     7

### Analyse de la cohérence
- Les heures hrmn dans caract-2024.csv apparaissent en bonne majorité valides, donc la dimension temporelle est fiable.
- Les coordonnées lat/long sont globalement cohérentes, ce qui est un bon point pour les analyses géographiques même si nous pouvons noter certaines anomalies
.
- an_nais présente des valeurs dans une plage cohérente pour des personnes, mais il faudra vérifier les très anciennes années si elles existent.
- vma devrait être numérique. Les valeurs non numériques ou les codes négatifs doivent être traitées comme anomalies.
- Les colonnes catégorielles comme sexe, grav, catv et catr contiennent parfois des codes -1 ou des modalités inattendues. Il faut discerner les véritables catégories des valeurs de données manquantes codées.

### Duplicates
Les jeux de données semblent bien structurés avec peu ou pas de doublons sur les clés logiques : Num_Acc pour caract; Num_Acc + voie pour lieux; Num_Acc + id_usager pour usagers; Num_Acc + id_vehicule pour vehicules.
Cela signifie que le lien entre les fichiers est exploitable sans devoir supprimer beaucoup de doublons.